In [1]:
import pandas as pd

In [2]:
mqm = pd.read_csv("../data/demo/mqm_final.csv")
ttp = pd.read_csv("../data/demo/ttp_final.csv")

In [3]:
import json

In [4]:
df = pd.concat([mqm, ttp], ignore_index=True)

In [5]:
def clean_motive(m):
    if pd.isna(m) or str(m).strip().lower() == "unknown":
        return None
    return str(m).strip()

In [7]:
def build_incident(row):
    return {
        "id": str(row["eventid"]),
        "date": f'{int(row["iyear"])}-{int(row["imonth"]):02d}-{int(row["iday"]):02d}',
        "city": row["city"],
        "actor": row["gname"],
        "attack_type": row["attacktype1_txt"],
        "weapon": row["weaptype1_txt"],
        "target_type": row["targtype1_txt"],
        "target": row["target1"],
        "nkill": None if pd.isna(row["nkill"]) else int(row["nkill"]),
        "nwound": None if pd.isna(row["nwound"]) else int(row["nwound"]),
        "motive": clean_motive(row["motive"]),
        "summary": row["summary"].strip(),
        "source": f'Global Terrorism Database (START), event ID {row["eventid"]}'
    }

In [8]:
graph = {"incidents": [build_incident(r) for _, r in df.iterrows()]}

In [11]:
with open("../data/demo/graphs/demo_graph.json", "w") as f:
    json.dump(graph, f, indent=2)

In [12]:
print(f"Total incidents: {len(graph['incidents'])}")
print(json.dumps(graph["incidents"][0], indent=2))

Total incidents: 27
{
  "id": "199803310001",
  "date": "1998-03-31",
  "city": "Karachi",
  "actor": "Muttahida Qami Movement (MQM)",
  "attack_type": "Bombing/Explosion",
  "weapon": "Explosives",
  "target_type": "Private Citizens & Property",
  "target": "Civilians at a market in Karachi, Pakistan",
  "nkill": 3,
  "nwound": 11,
  "motive": null,
  "summary": "03/31/1998: Suspected members of Muttahida Qaumi Movement-Haqiqi (MQM) used bombs to attack a crowded market in Karachi, Pakistan, killing three and wounding 11 others. The market sustained substantial damage.",
  "source": "Global Terrorism Database (START), event ID 199803310001"
}


In [15]:
with open("../data/demo/graphs/demo_kg.json", "r", encoding="utf-8") as f:
    kg = json.load(f)

print("=== METADATA ===")
print(json.dumps(kg["metadata"], indent=2, ensure_ascii=False))

print("\n=== SAMPLE NODES ===")
print(json.dumps(kg["nodes"][:5], indent=2, ensure_ascii=False)) 
 
print("\n=== SAMPLE EDGES ===")
print(json.dumps(kg["edges"][:5], indent=2, ensure_ascii=False))

=== METADATA ===
{
  "name": "Provenance-Aware Knowledge Graph for Security Event Analysis",
  "geographic_scope": "Sindh, Pakistan",
  "demo_scope": "Karachi",
  "source_dataset": "Global Terrorism Database",
  "provider": "START",
  "incident_count": 27
}

=== SAMPLE NODES ===
[
  {
    "id": "incident:199803310001",
    "type": "Incident",
    "properties": {
      "date": "1998-03-31",
      "nkill": 3,
      "nwound": 11
    }
  },
  {
    "id": "actor:Muttahida Qami Movement (MQM)",
    "type": "Actor",
    "name": "Muttahida Qami Movement (MQM)"
  },
  {
    "id": "date:1998-03-31",
    "type": "Date",
    "name": "1998-03-31"
  },
  {
    "id": "city:Karachi",
    "type": "City",
    "name": "Karachi"
  },
  {
    "id": "attack_type:Bombing/Explosion",
    "type": "AttackType",
    "name": "Bombing/Explosion"
  }
]

=== SAMPLE EDGES ===
[
  {
    "source": "actor:Muttahida Qami Movement (MQM)",
    "relation": "ASSOCIATED_WITH",
    "target": "incident:199803310001",
    "prove